In [34]:
from langchain_openai import ChatOpenAI 
import os
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model ="gpt-5-mini") 


### Tools

In [35]:
from langchain.tools import tool

In [36]:
@tool # Decorator to mark this function as a tool
def tool_duckduckgo_search(query: str) -> str:
    """Use this tool when you need to answer questions about current events or general knowledge questions."""
   
    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)
    return response

In [37]:
tool_duckduckgo_search.invoke("What is the capital of India?")

'This is a list of locations which have served as capital cities in India. The current capital city is New Delhi, which replaced Calcutta in 1911. India\'s administrative divisions of States and Union Territories and their capitals. India Standard Time (IST) now 9 hours and 30 minutes ahead of New York.New Delhi is the capital of India. Latitude: 28.62. Longitude: 77.21. Republic of India. Abbreviations: IN, IND. Capital: New Delhi. Edition. US UK Australia Canada Deutschland India Japan Latam. US residents can opt out of "sales" of personal data.Munich. 3. What is the capital of AUSTRALIA? Canberra.'

In [38]:
@tool # Decorator to mark this function as a tool
def tool_wikipedia_search(query: str) -> str:
    """Use this tool when you need to answer questions about persons, places etc."""
    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    response = wikipedia.run(query)
    return response

In [39]:
tool_wikipedia_search.invoke("Who is the president of USA?")

'Page: List of presidents of the United States\nSummary: The president of the United States is the head of state and head of government of the United States, indirectly elected to a four-year term via the Electoral College. Under the U.S. Constitution, the officeholder leads the executive branch of the federal government and is the commander-in-chief of the United States Armed Forces.\nThe first president, George Washington, won a unanimous vote of the Electoral College. The incumbent president is Donald Trump, who assumed office on January 20, 2025. Since the office was established in 1789, 45 men have served in 47 presidencies. The discrepancy is due to the nonconsecutive terms of Grover Cleveland (counted as the 22nd and 24th president) and Trump (counted as the 45th and 47th president).\nThe presidency of William Henry Harrison, who died 31 days after taking office in 1841, was the shortest in American history. Franklin D. Roosevelt served the longest, over twelve years, before dyi

In [40]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    print(result)

In [41]:
tool_arxiv_search.invoke("What are the latest research papers on deep learning?")

Published: 2023-01-03
Title: Deep Learning and Computational Physics (Lecture Notes)
Authors: Deep Ray, Orazio Pinti, Assad A. Oberai
Summary: These notes were compiled as lecture notes for a course developed and taught at the University of the Southern California. They should be accessible to a typical engineering graduate student with a strong background in Applied Mathematics.
  The main objective of these notes is to introduce a student who is familiar with concepts in linear algebra and partial differential equations to select topics in deep learning. These lecture notes exploit the strong connections between deep learning algorithms and the more conventional techniques of computational physics to achieve two goals. First, they use concepts from computational physics to develop an understanding of deep learning algorithms. Not surprisingly, many concepts in deep learning can be connected to similar concepts in computational physics, and one can utilize this connection to better un

In [42]:
@tool
def tool_personal_info(query: str) -> str:
    """Use this tool when you need to answer questions about personal information. Like name, age, occupation etc.
        Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    infos = [{
        "name": "John Doe",
        "age": 30,
        "Occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "Occupation": "Data Scientist"
    }]
    for info in infos:
        if query.lower() in info["name"].lower():
            return f"{info['name']} is a {info['age']} year old and works as a {info['Occupation']}."
    return "No information found."

In [43]:
tool_personal_info.invoke("John Doe")

'John Doe is a 30 year old and works as a Software Engineer.'

### Binding Tools

In [44]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info
    
]

In [45]:
llm_bind = llm.bind_tools(toolkit) #.bind_tools() is function that binds the tools to the llm. It takes a list of tools as input and returns a new llm that has access to those tools.

In [46]:
llm_bind.invoke("What is the age of John Doe? Make tool calls if necessary to answer the question.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 295, 'total_tokens': 385, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DW0MQHqKygMqanyrItydlDfQeNptv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da0db-ce9f-7881-a1bb-7dd0fd154708-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'query': 'John Doe'}, 'id': 'call_IdXJ2vskqJPrOU8WFwhC2LpO', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 295, 'output_tokens': 90, 'total_tokens': 385, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}})

### ReAct Agent

In [47]:
from langchain.agents import create_agent

my_agent = create_agent(llm_bind, toolkit)

In [48]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What is the age of John Doe? Make tool calls if necessary to answer the question."}]}
)

{'messages': [HumanMessage(content='What is the age of John Doe? Make tool calls if necessary to answer the question.', additional_kwargs={}, response_metadata={}, id='ae1277ee-3f33-4797-88a7-ab3aac8f13d5'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 295, 'total_tokens': 385, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DW0MU185TmHb3tKY4Hk0H9vGhmQzN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da0db-dfce-7940-9259-cb95bbceaee4-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'query': 'John Doe'}, 'id': 'call_b9DIhFkPA6YLxDj8rRdnfB1F', 'type': 'tool_call'}], invalid_tool